In [ ]:
""" SWE-Gym-data-filter.ipynb
This notebook filters the SWE-Gym instances

Output Json File:

instance_info.json      # Instance Information Dict, includes problem_statement, hints_text, touched-files-contents
instance_qa.json        # Instance QA Dict, currently has 10 instances as a  attempt
repo_dependency.json    # Repo Dependency Matrix, filtered by Cross-Commit-Consistency
file_dependency.json    # File Dependency Matrix (on the basis of Repo Dependency Matrix), filtered by Touched-Files

"""
from collections import Counter
import re
import json

import pandas as pd
from datasets import load_dataset

In [ ]:
DATASET_ID = "SWE-Gym/SWE-Gym"
CONFIG_NAME = None
SPLIT = "train"

load_kwargs = {"split": SPLIT}
if CONFIG_NAME is not None:
    load_kwargs["name"] = CONFIG_NAME

dataset = load_dataset(DATASET_ID, **load_kwargs)

In [ ]:
# Initialize logger and tqdm
from tqdm import tqdm

from constants.config import init_env_and_logger

logger = init_env_and_logger()

In [ ]:
print(f"Dataset: {DATASET_ID}")
print(f"Rows:    {dataset.num_rows:,}")
print(f"Columns: {len(dataset.column_names)}")

print("\nColumn names:")
for column in dataset.column_names:
    print(f"{column}", end="\t")

print("\nFeatures:")
dataset.features

In [ ]:
# == step 1 == analyze the base_commit feature of the datasets

# transform to pd
ds = dataset.to_pandas()
print(ds.iloc[0])

print("\n")

# base_commit numbers
bc = ds["base_commit"]
print(f"the lenth of origin base commit is {len(bc)}")

bc_filter = bc.unique()
print(f"the lenth of duplicated base commit is {len(bc_filter)}")

In [ ]:
# Find the "same base commit" instance

from collections import Counter
from textwrap import indent

bc_list = bc.tolist()
counter = Counter(bc_list)
same_bc = {bc : count for bc, count in counter.items() if count > 1}
print(same_bc.values())
print(len(same_bc))

In [ ]:
# tmp check for codex generated script

SAMPLE_ADDR = "../data/swe_gym_code_revert_5.jsonl"
with open(SAMPLE_ADDR, "r", encoding="utf-8") as f:
    instances = [json.loads(line) for line in f]
    print(json.dumps(instances[0], indent=4))

In [ ]:
# Find the None-Py_File in actual patch diff
# These instance may be filter

def touched_files(patch: str) -> list[str]:
    files = set()
    for match in re.finditer(r"^diff --git a/(.*?) b/(.*?)$", patch or "", re.M):
        old_path, new_path = match.groups()
        if old_path != new_path:
            continue
        files.add(new_path)
    return sorted(files)

patches = ds["patch"].tolist()
print(touched_files(patches[0]))

file_name_list = []
for patch in patches:
    file_name_list += touched_files(patch)

file_name_set = set(file_name_list)

def is_py(file_name: str) -> bool:
    return (file_name[-3:] == ".py") 
file_name_without_py = [file_name for file_name in file_name_set if is_py(file_name) == False]
None_py_files = [file_name for file_name in file_name_list if is_py(file_name) == False]

print(f'Number of Total patches: {len(patches)}')
print(f'Number of Total files: {len(file_name_list)}')
print(f'Number of Total duplicated files: {len(file_name_set)}')

print(file_name_without_py)
print(f'Number of Total Non-Py files: {len(None_py_files)}')
print(f'Number of Total duplicated Non-Py files: {len(file_name_without_py)}')

invalid_file = set() 
for invalid_file_name in file_name_without_py:
    for match in re.finditer(r".*\.(.*?)$", invalid_file_name):
        suf = match.group(1)
        invalid_file.add(suf)

print(invalid_file)

In [ ]:
# Analyze the touched file dependency in same repo

from collections import defaultdict
from pathlib import Path

rows = ds.to_dict("records")
repo_rows = defaultdict(list)
for row in rows:
    repo_name = row["repo"]
    repo_rows[repo_name].append(row)

REPO_BASE_ADDR = Path("../cloned_repo")

for _, instances in repo_rows.items():
    for instance in instances:
        touched_file = touched_files(instance["patch"])
        instance["touched_file"] = touched_file

print(repo_rows.keys())


In [ ]:
# Build a per-repo target x middle consistency matrix.
# matrix[target_idx][middle_idx] == 1 means every file touched by the middle
# instance is identical between target.base_commit and middle.base_commit.

import subprocess
from functools import lru_cache

def resolve_repo_dir(repo_key: str, instances: list[dict], repo_base_addr: Path) -> Path:
    repo_full_name = instances[0]["repo"]
    candidates = [
        repo_base_addr / repo_key,
        repo_base_addr / repo_full_name.replace("/", "__"),
        repo_base_addr / repo_full_name,
        repo_base_addr / repo_full_name.split("/")[-1],
    ]

    for candidate in candidates:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        f"Cannot find a cloned git repo for {repo_full_name}. "
        f"Expected one of: {[str(path) for path in candidates]}"
    )


def run_git(repo_dir: Path, *args: str, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(
        ["git", "-C", str(repo_dir), *args],
        check=check,
        capture_output=True,
        text=True,
    )

from enum import Enum
class Relation (Enum):
    OLD = 1
    NEW = 2
    NR = 3

def compare_commit_ancestry(
    repo_dir: Path, first_commit: str, second_commit: str
) -> str:
    """Compare two commits and return their ancestor relationship.

    Identical commits are reported as '前一个是祖先'.
    """
    first_is_ancestor = run_git(
        repo_dir,
        "merge-base",
        "--is-ancestor",
        first_commit,
        second_commit,
        check=False,
    )
    if first_is_ancestor.returncode == 0:
        return Relation.OLD
    if first_is_ancestor.returncode != 1:
        raise RuntimeError(
            f"Failed to compare commits {first_commit} and {second_commit}: "
            f"{first_is_ancestor.stderr.strip()}"
        )

    second_is_ancestor = run_git(
        repo_dir,
        "merge-base",
        "--is-ancestor",
        second_commit,
        first_commit,
        check=False,
    )
    if second_is_ancestor.returncode == 0:
        return Relation.NEW
    if second_is_ancestor.returncode == 1:
        return Relation.NR

    raise RuntimeError(
        f"Failed to compare commits {first_commit} and {second_commit}: "
        f"{second_is_ancestor.stderr.strip()}"
    )

def unique_preserve_order(items: list[str]) -> list[str]:
    return list(dict.fromkeys(items))

@lru_cache(maxsize=None)
def middle_files_consistent(
    repo_dir_str: str,
    target_base_commit: str,
    middle_base_commit: str,
    middle_files: tuple[str, ...],
) -> int:
    repo_dir = Path(repo_dir_str)
    if not middle_files:
        return 0

    result = run_git(
        repo_dir,
        "diff",
        "--quiet",
        target_base_commit,
        middle_base_commit,
        "--",
        *middle_files,
        check=False,
    )

    if result.returncode == 0:
        return 1
    if result.returncode == 1:
        return 0

    raise RuntimeError(
        f"git diff failed for repo={repo_dir}, target={target_base_commit}, "
        f"middle={middle_base_commit}, files={middle_files}. stderr={result.stderr.strip()}"
    )

In [ ]:


repo_target_middle_consistency: dict[str, dict[str, dict[str, int]]] = {}

for repo_name, instances in repo_rows.items():
    # Test for modin-project/modin first
    if repo_name != "modin-project/modin":
        continue

    logger.info(f"Now Processing the repo {repo_name}")
    
    try:
        repo_dir = resolve_repo_dir(repo_name, instances, REPO_BASE_ADDR)
    except:
        logger.warning(f"The ropo {repo_name} doesn't exist in cloned_repo.")
        continue

    matrix = defaultdict(dict)

    pbar = tqdm(instances)
    for target in pbar:

        target_base_commit = target["base_commit"]
        target_id = target["instance_id"]
        
        pbar.set_description(f"Now Process the target_instance {target_id}")

        row = defaultdict(int) 

        for middle in instances:
            middle_id = middle["instance_id"]

            if target_id == middle_id:
                row[middle_id] = 0
                continue

            middle_base_commit = middle["base_commit"]
            middle_files = tuple(unique_preserve_order(middle["touched_file"]))
            row[middle_id] = middle_files_consistent(
                        str(repo_dir),
                        target_base_commit,
                        middle_base_commit,
                        middle_files, 
                    )

        matrix[target_id] = row

    repo_target_middle_consistency[repo_name] = matrix

from constants.addr import REPO_DEPENDENCY_ADDR
with open(REPO_DEPENDENCY_ADDR, "w", encoding = "utf-8") as f:
    json.dump(repo_target_middle_consistency, f)




In [ ]:
from constants.addr import load_json, save_json, REPO_DEPENDENCY_ADDR

repo_target_middle_consistency = load_json(REPO_DEPENDENCY_ADDR)
consistency_and_different_files = repo_target_middle_consistency.copy()

for repo_name, matrix in consistency_and_different_files.items():
    print(f"repo_name:{repo_name}, size:{len(matrix)}")
    repo_dict = {instance["instance_id"] : instance for instance in repo_rows[repo_name]}
    for target_id, middle_dict in matrix.items():
        tot = 0
        target_files = set(repo_dict[target_id]["touched_file"])
        for middle_id, flag in middle_dict.items():
            if flag == 0:
                continue
            else:
                middle_files = set(repo_dict[middle_id]["touched_file"])
                # print(f"middle_files are {middle_files}")
                # print(f"target_files are {target_files}")
                if middle_files.isdisjoint(target_files) == False:
                    consistency_and_different_files[repo_name][target_id][middle_id] = 0
                    # print(f"Find a non-disjoint file between {target_id} and {middle_id}.")
                    # print(f"The target-touched-files are {target_files}.")
                    # print(f"The middle-touched-files are {middle_files}.")
                else:
                    tot += flag
        # print(f"{target_id}: {tot}")

from constants.addr import FILE_DEPENDENCY_ADDR
save_json(FILE_DEPENDENCY_ADDR, consistency_and_different_files)


In [ ]:
target_invalid_number = defaultdict(dict)

# Verify the difference

from constants.addr import load_json

repo_target_middle_consistency = load_json(REPO_DEPENDENCY_ADDR)
consintency_and_different_files = load_json(FILE_DEPENDENCY_ADDR)

for repo_name, matrix in repo_target_middle_consistency.items():
    for target_id, middle_dict in matrix.items():
        tot = 0
        for middle_id, flag in middle_dict.items():
            tot += flag
        # print(f"{target_id}: {tot}")
        target_invalid_number[repo_name][target_id] = tot

for repo_name, matrix in consistency_and_different_files.items():
    for target_id, middle_dict in matrix.items():
        tot = 0
        for middle_id, flag in middle_dict.items():
            tot += flag
        # print(f"{target_id}: {tot}")
        if tot != target_invalid_number[repo_name][target_id]:
            print(f"difference between {target_id}")

In [ ]:
# To test a target_repo
# cnt = 0
# target_cnt = 3

# least_num is the least_num of valid middle instance
least_num = 8
valid_cnt = 0

for repo_name, matrix in consistency_and_different_files.items():
    # cnt += 1
    # if cnt != target_cnt:
    #     continue
    # print(f"repo:{repo_name}, size:{len(matrix)}")
    for target_id, middle_dict in matrix.items():
        tot = 0
        for middle_id, flag in middle_dict.items():
            tot += flag
        if tot >= least_num :
            valid_cnt+=1

print(valid_cnt)
    

In [ ]:
# for repo_name in consistency_and_different_files.keys():
#     print(repo_name.split('/')[1],end=" ")

BASE_ADDR = "../cloned_repo/"
repo_dirs = {
    repo_name : BASE_ADDR + repo_name.split('/')[1]
    for repo_name in consistency_and_different_files
}

# repo_dirs

In [ ]:
# Analyse the Branch Feature of each Target instance

cf = consistency_and_different_files


@lru_cache(maxsize=None)
def cached_commit_relation(
    repo_dir_str: str, first_commit: str, second_commit: str
) -> Relation:
    return compare_commit_ancestry(Path(repo_dir_str), first_commit, second_commit)


def build_descendant_chains(
    repo_dir: Path,
    target_id: str,
    middle_ids: list[str],
    repo_dict: dict[str, dict],
) -> list[list[str]]:
    """Build maximal target-to-descendant paths from eligible middle instances.

    A branch produces multiple paths. A merge is included in every path that
    reaches it. Instances sharing one base commit are kept together in a
    deterministic order.
    """

    # From commit to ids for middle instance id
    commit_to_ids: dict[str, list[str]] = {}

    for middle_id in middle_ids:
        middle_commit = repo_dict[middle_id]["base_commit"]

        # When middle commit is the same as target, that means this middle instance is an available instance for tasks sequence.
        # if  middle_commit == target_commit:
        #     continue
        commit_to_ids.setdefault(middle_commit, []).append(middle_id)

    # Non middle instance available
    if not commit_to_ids:
        return []

    # commits: dict from commit to ids
    commits = sorted(commit_to_ids)

    # children: immediate child for each middle
    children: dict[str, set[str]] = {commit: set() for commit in commits}

    # Keep only immediate ancestor-to-descendant edges (the transitive reduction).
    for older_commit in commits:
        # For each commit, take it as a older, find the descendant commit
        descendants = [
            newer_commit for newer_commit in commits
            if newer_commit != older_commit
            and cached_commit_relation(
                str(repo_dir), older_commit, newer_commit
            ) == Relation.OLD
        ]

        # For each descendant commit, if there isn't any intermediate commit, put it into the children of ancestor
        for newer_commit in descendants:
            has_intermediate = any(
                intermediate_commit != newer_commit
                and cached_commit_relation(
                    str(repo_dir), intermediate_commit, newer_commit
                ) == Relation.OLD
                for intermediate_commit in descendants
            )
            if not has_intermediate:
                children[older_commit].add(newer_commit)

    # child_commits are commits_set which can be a child of another commit
    child_commits = {child for values in children.values() for child in values}

    # The Target's children
    roots = [commit for commit in commits if commit not in child_commits]

    # Find the longest path for a target's child
    def paths_from(commit: str) -> list[list[str]]:
        if not children[commit]:
            return [[commit]]
        return [
            [commit, *suffix]
            for child in sorted(children[commit])
            for suffix in paths_from(child)
        ]

    return [
        [
            target_id,
            # First: Get the commit list as the paths_from do (return the commit_path)
            # Then : Transform the commit list to id list
            *(
                instance_id
                for commit in commit_path
                for instance_id in sorted(commit_to_ids[commit])
            ),
        ]
        for root in roots
        for commit_path in paths_from(root)
    ]


branch_chains: dict[str, dict[str, list[list[str]]]] = {}

repo_bar = tqdm(cf.items(), desc="Dealing all repos.", position = 0)
for repo_name, matrix in repo_bar:
    # id2instance map
    repo_dict = {instance["instance_id"] : instance for instance in repo_rows[repo_name]}

    # Find the repo_dir
    try:
        repo_dir = repo_dirs[repo_name]
    except Exception as e:
        logger.warning(str(e))
        continue

    # for each target, find the middle instances
    target_bar = tqdm(matrix.items(), desc=f"Now dealing with {repo_name} targets.", leave=False, position = 1)
    for target_id, middle_dict in target_bar:
        target_commit = repo_dict[target_id]["base_commit"] 
        
        # Available middle list
        middle_list = [middle_id for middle_id, flag in middle_dict.items() if flag == 1]

        # Keep only proper descendants of the target commit.
        middle_list = [
            middle_id for middle_id in middle_list
            if compare_commit_ancestry(
                repo_dir, target_commit,
                repo_dict[middle_id]["base_commit"]
            ) == Relation.OLD
        ]
    
        # build the chains
        chains = build_descendant_chains(
            Path(repo_dir), target_id, middle_list, repo_dict
        )

        # Add it to the dict
        if chains:
            branch_chains.setdefault(repo_name, {})[target_id] = chains

# Fail due to ipynb core doesn't update the constants/addr packages states
# from constants.addr import TARGET_SEQUENCE_ADDR
# save_json(TARGET_SEQUENCE_ADDR, branch_chains)
            


In [ ]:
TARGET_SEQUENCE_ADDR = "../data/target_sequence.json"
save_json(TARGET_SEQUENCE_ADDR, branch_chains)

for repo_name , target_dict in branch_chains.items():
    for target_id, middle_list in target_dict.items():
        if len(middle_list) > 1:
            print("Find a muli list!")
# 666, just 4 list has multiple branches
# I just analyse like a foolman

In [ ]:
# Get the instance touched files for llm-QA-generation

instance_info_list = []

for repo_name,instances in repo_rows.items():
    print(repo_name)

    try:
        repo_dir = resolve_repo_dir(repo_name, instances, REPO_BASE_ADDR)
    except FileNotFoundError as exc:
        logger.warning(str(exc))
        continue

    pbar = tqdm(instances, desc = f"processing the repo {repo_name}")
    for instance in pbar:
        instance_info = {
            "instance_id": instance["instance_id"],
            "problem_statement": instance["problem_statement"],
            "hints_text": instance["hints_text"],
            "files_list": []
        }
        touched_file = unique_preserve_order(instance["touched_file"])
        base_commit = instance["base_commit"]

        for file_path in touched_file:
            result = run_git(
                repo_dir,
                "show",
                f"{base_commit}:{file_path}",
                check=False,
            )

            if result.returncode != 0:
                logger.warning(
                    "Failed to load %s at %s for %s: %s",
                    file_path,
                    base_commit,
                    instance["instance_id"],
                    result.stderr.strip(),
                )
                continue

            instance_info["files_list"].append(result.stdout)

        instance_info_list.append(instance_info)

In [ ]:
print(instance_info_list[0]["files_list"])

from constants.addr import INSTANCE_INFO_ADDR
with open(INSTANCE_INFO_ADDR, "w", encoding="utf-8") as f:
    json.dump(instance_info_list, f)

from typing_extensions import override
from litellm import OpenAI
from dotenv import load_dotenv
import os

# ===== load dotenv ======
load_dotenv(dotenv_path="./.env", override=True)

# ===== Initialize Client =====
client = OpenAI(
    base_url = os.getenv("API_BASE_URL"),
    api_key = os.getenv("API_KEY")
)

model = os.getenv("API_MODEL")
print(model)

def deal_response(response):
    return response.choices[0].message.content.strip()

# Test AK
# response = client.chat.completions.create(
#     model=model,
#     messages=[{
#         "role": "user",
#         "content": "hi, how are you today"
#     }]
# )

# print(deal_response(response))

test_cnt = 10
from constants.addr import INSTANCE_QA_ADDR
instance_qa = load_json(INSTANCE_QA_ADDR)
for instance in instance_info_list[:test_cnt]:
    if instance["instance_id"] in instance_qa and len(instance_qa[instance["instance_id"]]) > 0:
        continue
    instance_qa[instance["instance_id"]] = {}
    prompt = f""" 
You are a professional code analysis assistant, you are good at generating high quality questions about code repository.
Now you are going to generate QA-pairs to test another coding agent memory ability.

Given:
You are given one SWE-Gym instance. The instance contains problem_statement, hints_text, and files_list. files_list contains the exact source-code contents of the files touched by the target fix at the base commit. In the evaluation, another coding agent will first solve the issue. Later, after the touched files have been deleted from the workspace, the agent will be asked questions about those deleted files. Therefore, each generated question must be answerable from the agent's memory of the file contents it needed to inspect while solving the issue.

Tasks:
Generate high-quality QA pairs about the code in files_list. The questions should focus on code details that are likely to be read during a realistic debugging and fixing trajectory for the given problem_statement and hints_text. Prefer questions about relevant functions, classes, methods, control flow, data transformations, edge-case handling, constants, imports, helper APIs, or interactions between code paths that matter for understanding or fixing the issue.

Rules:
1. Every answer must be fully supported by files_list. Do not use external knowledge, repository state outside files_list, or assumptions about the final patch.
2. The question must not be answerable from problem_statement or hints_text alone. It must require specific code content from files_list.
3. The question must be relevant to the likely fix trajectory. Avoid arbitrary trivia such as unrelated imports, formatting, comments, or line-order details unless they are directly useful for solving the problem.
4. Ask questions that remain meaningful after the file is deleted. The question should refer to stable code concepts such as function names, class names, parameters, branches, or behavior, not to line numbers.
5. Each gold answer should be concise but specific enough to grade automatically or manually. Include the key identifiers or behavior needed to distinguish a correct answer from a vague guess.
6. If files_list does not contain enough relevant code to produce grounded QA pairs, generate fewer pairs rather than inventing unsupported content.Quality is more important than Quantity.
7. Output valid JSON only. Do not include markdown, explanations, comments, or any text outside the JSON object. The Questions and Gold_answers arrays must have the same length, and item i in Gold_answers must answer item i in Questions.

Input:
- problem_statement:{instance["problem_statement"]}
- hints_text:{instance["hints_text"]}
- files_list:{instance["files_list"]}

Output_json_format:
{{
"Questions": [],
"Gold_answers": []
}}

"""


    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{
                "role": "user",
                "content": prompt
            }]
        )
        dr = deal_response(response)
        qa = json.loads(dr)
        instance_qa[instance["instance_id"]]["questions"] = qa["Questions"]
        instance_qa[instance["instance_id"]]["gold_answers"] = qa["Gold_answers"]
    except json.JSONDecodeError as json_err:
        logger.warning(f"Instance {instance["instance_id"]} QA-Generation failed due to json_Decoder_failed.") 
        continue
    except Exception as e:
        logger.warning(f"Instance {instance["instance_id"]} QA-Generation failed.") 
        continue


logger.info("QA-Generation complete")

In [ ]:
# Dump the dict to json
with open(INSTANCE_QA_ADDR, "w", encoding = "utf-8") as f:
    json.dump(instance_qa,f)
print(json.dumps(instance_qa, indent=4))